# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [14]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

#检查环境和API_KEY
# Load environment variables from .env
load_dotenv()

# Check and print results
# doublecheck_env("example.env")
doublecheck_env(".env")

OPENAI_API_KEY=****ywsA
DEEPSEEK_API_KEY=****c1e4
DEEPSEEK_BASE_URL=****.com
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [15]:
from langchain_community.utilities import SQLDatabase
#把本地的 Chinook.db SQLite 数据库包装成一个 LangChain 可用的 SQLDatabase 对象。
#通过uri创建数据库对象
#sqlite:///表示是本地SQLite文件
#后面的Chinook.db就是数据库文件名
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
#最后将本地数据库文件打包装饰为能够通信的数据库对象

In [16]:
from dataclasses import dataclass
#这里的dataclass是一个官方的装饰器
#用来打包类,直接加上__init__初始化和__repr__打印等方法
@dataclass
class RuntimeContext:
    db: SQLDatabase

In [17]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

#打包为tool
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    #从上下文对象中获取数据库对象
    #不过为什么是context.db?
    #agent启动时会自动创建runtime,并且有context,store等属性
    #从下面可以看到,在agent启动时有context=RuntimeContext(db=db)
    #此时runtime的context就变为了RuntimeContext,其中有db数据库对象
    #而get_runtime函数中的RuntimeContext则只是声明runtime的context对象是RuntimeContext类型
    #于是整个流程就是:agent启动时注入上下文,执行工具时使用get_runtime函数获取上下文(并声明类型),获取这次运行的runtime后获取上下文中的db对象
    db = runtime.context.db

    try:
        #使用数据库对象尝试执行请求(query)
        return db.run(query)
    except Exception as e:
        #遇到错误就抛出
        return f"Error: {e}"

In [18]:
SYSTEM_PROMPT = """你是一名严谨的 SQLite 数据分析助手。

规则：
- 请逐步思考。
- 当你需要数据时，只能调用工具 `execute_sql`，并且一次只执行一条 SELECT 查询。
- 只允许只读操作；禁止使用 INSERT、UPDATE、DELETE、ALTER、DROP、CREATE、REPLACE、TRUNCATE。
- 除非用户明确要求，否则结果最多返回 5 行。
- 如果工具返回以 `Error:` 开头的错误信息，请修正 SQL 后再重试。
- 优先使用明确的列名，避免使用 SELECT *。
"""

In [19]:
from langchain.agents import create_agent

agent = create_agent(
    # model="openai:gpt-5",
    model="deepseek-chat",
    #工具
    tools=[execute_sql],
    #提示词
    system_prompt=SYSTEM_PROMPT,
    #上下文schema,定义上下文结构
    context_schema=RuntimeContext,
)

## Repeated Queries

In [20]:
question = "This is Frank Harris, What was the total on my last invoice?response with Chinese."
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    #这里的列表是为了出现错误时方便查找错误,在案例中没有实际用途
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?response with Chinese.
================================== Ai Message ==================================

我来帮您查询Frank Harris的最后一张发票的总金额。首先，我需要找到Frank Harris的客户信息，然后查询他的最后一张发票。
Tool Calls:
  execute_sql (call_00_sArl2bRhTjiSPjdJ3JTE4D2l)
 Call ID: call_00_sArl2bRhTjiSPjdJ3JTE4D2l
  Args:
    query: SELECT CustomerId, FirstName, LastName FROM customers WHERE FirstName = 'Frank' AND LastName = 'Harris'
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: customers
[SQL: SELECT CustomerId, FirstName, LastName FROM customers WHERE FirstName = 'Frank' AND LastName = 'Harris']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

让我先查看数据库中有哪些表：
Tool Calls:
  execute_sql (c

In [10]:
question = "What were the titles?response with Chinese."

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

我需要先了解您询问的是哪个表格中的标题信息。您能提供更多上下文吗？比如：

1. 您指的是哪个数据库中的哪个表格？
2. 您想查询什么类型的标题？比如文章标题、电影标题、书籍标题等？

或者，我可以先查看一下数据库中有哪些表格，然后您再告诉我具体想查询哪个表格的标题信息。
Tool Calls:
  execute_sql (call_00_JkODkwEMnlQ053sQJXk8aLBW)
 Call ID: call_00_JkODkwEMnlQ053sQJXk8aLBW
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

我看到数据库中有多个表格。让我查看一下这些表格的结构，以便了解哪个表格包含标题信息：
Tool Calls:
  execute_sql (call_00_AN58g3AWZcXU4PFZIcM1sbvN)
 Call ID: call_00_AN58g3AWZcXU4PFZIcM1sbv

## Add memory

In [11]:
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    # model="openai:gpt-5",
    model="deepseek-chat",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [13]:
question = "This is Frank Harris, What was the total on my last invoice?response with Chinese."
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?response with Chinese.
================================== Ai Message ==================================

我来帮您查询Frank Harris的最后一张发票的总额。首先，我需要查看数据库中有哪些表，然后找到与发票相关的信息。
Tool Calls:
  execute_sql (call_00_zPfCQOcsS4mGLVfs08isLgzK)
 Call ID: call_00_zPfCQOcsS4mGLVfs08isLgzK
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================

我看到有一个`Invoice`表和一个`Customer`表。首先让我查找Frank Harris的客户信息。
Tool Calls:
  execute_sql (call_00_ulN4pvXg3Y8Sdrh5GS7j3GKz)
 Call ID: call_00_ulN4pvXg3Y8Sdrh5GS7j3GKz


In [12]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_NVIzXxF2vjAD1uAi2kI3CXl4)
 Call ID: call_NVIzXxF2vjAD1uAi2kI3CXl4
  Args:
    query: SELECT il.InvoiceLineId, t.Name AS Title
FROM InvoiceLine AS il
JOIN Track AS t ON t.TrackId = il.TrackId
WHERE il.InvoiceId = 374
ORDER BY il.InvoiceLineId
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(2021, 'Holier Than Thou'), (2022, 'Through The Never'), (2023, 'My Friend Of Misery'), (2024, 'The Wait'), (2025, 'Blitzkrieg')]
================================== Ai Message ==================================

The titles on your last invoice (ID 374) include:
- Holier Than Thou
- Through The Never
- My Friend Of Misery
- The Wait
- Blitzkrieg

Would you like the full list with quantities and prices?


## Try your own queries
Now that there is memory, check the agents recall!

In [ ]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)